# GraphRAG with Milvus：只用向量库做“多跳关系推理”

这节解决的问题：**传统 RAG 很难稳定回答 multi-hop（多跳）问题**，因为关键关系往往分散在不同段落里，单次相似度检索容易漏掉“逻辑上必须的中间关系”。

核心思路（原仓库的主线）：

1. 把语料拆成三类向量集合（Milvus collections）
   - **entities**（实体，节点）
   - **relationships**（关系三元组拼成一句话，边）
   - **passages**（原文段落，用作最终证据 + 和 naive RAG 对照）
2. 查询时走两条路并行召回：**实体路** + **关系路**
3. 用邻接矩阵做 **subgraph expansion（扩展 1-hop / 2-hop）** 生成候选关系
4. 用 LLM **rerank** 候选关系，选出最有用的 3 条
5. 把关系映射回原文 passages，再让 LLM 生成答案（或对照 naive RAG 的 passages）


## 0) 导入依赖并加载环境变量

请自行确保已安装：`pymilvus`、`numpy`、`scipy`、`tqdm`、`langchain-openai`。


In [ ]:
import os
from collections import defaultdict
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from scipy.sparse import csr_matrix
from tqdm import tqdm

from pymilvus import MilvusClient

from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

load_dotenv()

DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import os
os.environ["HTTP_PROXY"] = "http://127.0.0.1:7890"
os.environ["HTTPS_PROXY"] = "http://127.0.0.1:7890"
os.environ["ALL_PROXY"] = "http://127.0.0.1:7890"

## 1) 连接 Milvus + 初始化 LLM / Embeddings

这里用 `MilvusClient`。如果你本地没有 Milvus 服务，可以用 **Milvus Lite** 的方式：把 `uri` 指向一个本地 db 文件路径（pymilvus 支持）。


In [ ]:
MILVUS_URI = str((DATA_DIR / "graphrag_milvus.db").resolve())

milvus_client = MilvusClient(uri=MILVUS_URI)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

# text-embedding-3-small 的维度是 1536（固定常量）
EMBEDDING_DIM = 1536

## 2) 离线数据：passages + triplets（nano dataset）

为了把 GraphRAG 的“扩展 + rerank”讲清楚，这里直接用一个小数据集：

- 每条样本有一个 passage
- 以及一组 triplets：`[subject, predicate, object]`


In [ ]:
nano_dataset = [
    {
        "passage": "Jakob Bernoulli (1654–1705): Jakob was one of the earliest members of the Bernoulli family to gain prominence in mathematics. He made significant contributions to calculus, particularly in the development of the theory of probability. He is known for the Bernoulli numbers and the Bernoulli theorem, a precursor to the law of large numbers. He was the older brother of Johann Bernoulli, another influential mathematician, and the two had a complex relationship that involved both collaboration and rivalry.",
        "triplets": [
            ["Jakob Bernoulli", "made significant contributions to", "calculus"],
            ["Jakob Bernoulli", "made significant contributions to", "the theory of probability"],
            ["Jakob Bernoulli", "is known for", "the Bernoulli numbers"],
            ["Jakob Bernoulli", "is known for", "the Bernoulli theorem"],
            ["The Bernoulli theorem", "is a precursor to", "the law of large numbers"],
            ["Jakob Bernoulli", "was the older brother of", "Johann Bernoulli"],
        ],
    },
    {
        "passage": "Johann Bernoulli (1667–1748): Johann, Jakob’s younger brother, was also a major figure in the development of calculus. He worked on infinitesimal calculus and was instrumental in spreading the ideas of Leibniz across Europe. Johann also contributed to the calculus of variations and was known for his work on the brachistochrone problem, which is the curve of fastest descent between two points.",
        "triplets": [
            ["Johann Bernoulli", "was a major figure of", "the development of calculus"],
            ["Johann Bernoulli", "was", "Jakob's younger brother"],
            ["Johann Bernoulli", "worked on", "infinitesimal calculus"],
            ["Johann Bernoulli", "was instrumental in spreading", "Leibniz's ideas"],
            ["Johann Bernoulli", "contributed to", "the calculus of variations"],
            ["Johann Bernoulli", "was known for", "the brachistochrone problem"],
        ],
    },
    {
        "passage": "Daniel Bernoulli (1700–1782): The son of Johann Bernoulli, Daniel made major contributions to fluid dynamics, probability, and statistics. He is most famous for Bernoulli’s principle, which describes the behavior of fluid flow and is fundamental to the understanding of aerodynamics.",
        "triplets": [
            ["Daniel Bernoulli", "was the son of", "Johann Bernoulli"],
            ["Daniel Bernoulli", "made major contributions to", "fluid dynamics"],
            ["Daniel Bernoulli", "made major contributions to", "probability"],
            ["Daniel Bernoulli", "made major contributions to", "statistics"],
            ["Daniel Bernoulli", "is most famous for", "Bernoulli’s principle"],
            ["Bernoulli’s principle", "is fundamental to", "the understanding of aerodynamics"],
        ],
    },
    {
        "passage": "Leonhard Euler (1707–1783) was one of the greatest mathematicians of all time, and his relationship with the Bernoulli family was significant. Euler was born in Basel and was a student of Johann Bernoulli, who recognized his exceptional talent and mentored him in mathematics. Johann Bernoulli’s influence on Euler was profound, and Euler later expanded upon many of the ideas and methods he learned from the Bernoullis.",
        "triplets": [
            ["Leonhard Euler", "had a significant relationship with", "the Bernoulli family"],
            ["Leonhard Euler", "was born in", "Basel"],
            ["Leonhard Euler", "was a student of", "Johann Bernoulli"],
            ["Johann Bernoulli's influence", "was profound on", "Euler"],
        ],
    },
]

len(nano_dataset)

4

## 3) 从 triplets 构建：entities / relations / passages + 邻接映射

- entity = triplet 的 subject/object 去重
- relation = 把三元组拼成一句自然语言（便于 embedding）
- 同时建立两张“图连接表”
  - `entityid_2_relationids`: entity_id → 它参与的 relation_id 列表
  - `relationid_2_passageids`: relation_id → 出现在哪些 passage_id


In [ ]:
entityid_2_relationids: dict[int, list[int]] = defaultdict(list)
relationid_2_passageids: dict[int, list[int]] = defaultdict(list)

entities: list[str] = []
relations: list[str] = []
passages: list[str] = []

for passage_id, dataset_info in enumerate(nano_dataset):
    passage, triplets = dataset_info["passage"], dataset_info["triplets"]
    passages.append(passage)
    for s, p, o in triplets:
        if s not in entities:
            entities.append(s)
        if o not in entities:
            entities.append(o)

        relation = " ".join([s, p, o])
        if relation not in relations:
            relations.append(relation)
            rid = len(relations) - 1
            entityid_2_relationids[entities.index(s)].append(rid)
            entityid_2_relationids[entities.index(o)].append(rid)

        relationid_2_passageids[relations.index(relation)].append(passage_id)

len(entities), len(relations), len(passages)

(25, 22, 4)

## 4) 创建 3 个 Milvus collections 并写入向量

我们把三类文本分别写入三个 collection：

- `entity_collection`
- `relation_collection`
- `passage_collection`


In [ ]:
def create_milvus_collection(collection_name: str) -> None:
    if milvus_client.has_collection(collection_name=collection_name):
        milvus_client.drop_collection(collection_name=collection_name)

    milvus_client.create_collection(
        collection_name=collection_name,
        dimension=EMBEDDING_DIM,
        consistency_level="Strong",
    )


entity_col_name = "entity_collection"
relation_col_name = "relation_collection"
passage_col_name = "passage_collection"

create_milvus_collection(entity_col_name)
create_milvus_collection(relation_col_name)
create_milvus_collection(passage_col_name)

In [ ]:
def milvus_insert(collection_name: str, text_list: list[str]) -> None:
    batch_size = 128
    for row_id in tqdm(range(0, len(text_list), batch_size), desc=f"Inserting {collection_name}"):
        batch_texts = text_list[row_id : row_id + batch_size]
        batch_embeddings = embedding_model.embed_documents(batch_texts)

        batch_ids = [row_id + j for j in range(len(batch_texts))]
        batch_data = [
            {"id": id_, "text": text, "vector": vector}
            for id_, text, vector in zip(batch_ids, batch_texts, batch_embeddings)
        ]
        milvus_client.insert(collection_name=collection_name, data=batch_data)


milvus_insert(relation_col_name, relations)
milvus_insert(entity_col_name, entities)
milvus_insert(passage_col_name, passages)

Inserting passage_collection: 100%|██████████| 1/1 [00:00<00:00,  1.27it/s]


## 5) 在线查询：双路召回（实体路 + 关系路）

原 notebook 里这里用 NER 先抽 query 里的实体。为了聚焦 GraphRAG 主流程，这里先**手写 NER 结果**。


In [ ]:
query = "What contribution did the son of Euler's teacher make?"

query_ner_list = ["Euler"]
# query_ner_list = ner(query)  # In practice, replace it with your own NER

query_ner_embeddings = [embedding_model.embed_query(x) for x in query_ner_list]
query_embedding = embedding_model.embed_query(query)

top_k = 3

entity_search_res = milvus_client.search(
    collection_name=entity_col_name,
    data=query_ner_embeddings,
    limit=top_k,
    output_fields=["id"],
)

relation_search_res = milvus_client.search(
    collection_name=relation_col_name,
    data=[query_embedding],
    limit=top_k,
    output_fields=["id"],
)[0]

entity_search_res, relation_search_res[:2]

(data: [[{'id': 24, 'distance': 0.0, 'entity': {'id': 24}}, {'id': 20, 'distance': 0.21836066246032715, 'entity': {'id': 20}}, {'id': 0, 'distance': 0.5901350378990173, 'entity': {'id': 0}}]],
 [{'id': 21, 'distance': 0.3445613980293274, 'entity': {'id': 21}},
  {'id': 20, 'distance': 0.3635290861129761, 'entity': {'id': 20}}])

## 6) Subgraph Expansion：用邻接矩阵扩展 1-hop / 2-hop 候选关系

关键点：

- `entity_relation_adj` 表示 entity 与 relation 是否相连
- 用矩阵乘法得到 entity-entity、relation-relation 的邻接
- 最终从“命中的实体/关系”扩展出候选关系集合


In [ ]:
entity_relation_adj = np.zeros((len(entities), len(relations)))
for entity_id in range(len(entities)):
    entity_relation_adj[entity_id, entityid_2_relationids[entity_id]] = 1

entity_relation_adj = csr_matrix(entity_relation_adj)

entity_adj_1_degree = entity_relation_adj @ entity_relation_adj.T
relation_adj_1_degree = entity_relation_adj.T @ entity_relation_adj

target_degree = 1
entity_adj_target_degree = entity_adj_1_degree
for _ in range(target_degree - 1):
    entity_adj_target_degree = entity_adj_target_degree @ entity_adj_1_degree.T

relation_adj_target_degree = relation_adj_1_degree
for _ in range(target_degree - 1):
    relation_adj_target_degree = relation_adj_target_degree @ relation_adj_1_degree.T

entity_relation_adj_target_degree = entity_adj_target_degree @ entity_relation_adj

In [ ]:
expanded_relations_from_relation: set[int] = set()
expanded_relations_from_entity: set[int] = set()

hit_relation_ids = [r["entity"]["id"] for r in relation_search_res]
for rid in hit_relation_ids:
    expanded_relations_from_relation.update(
        relation_adj_target_degree[rid].nonzero()[1].tolist()
    )

hit_entity_ids = [
    one_entity_res["entity"]["id"]
    for one_entity_search_res in entity_search_res
    for one_entity_res in one_entity_search_res
]
for eid in hit_entity_ids:
    expanded_relations_from_entity.update(
        entity_relation_adj_target_degree[eid].nonzero()[1].tolist()
    )

relation_candidate_ids = list(expanded_relations_from_relation | expanded_relations_from_entity)
relation_candidate_texts = [relations[rid] for rid in relation_candidate_ids]

len(relation_candidate_ids), relation_candidate_ids[:10]

(16, [0, 1, 2, 3, 5, 6, 7, 8, 9, 10])

## 7) LLM Reranking：从候选关系中选最有用的 3 条

这里复用原 notebook 的 one-shot 思路，但用 LangChain 的 **结构化输出**来拿到稳定的 JSON。


In [ ]:
class RerankResult(BaseModel):
    thought_process: str = Field(description="Why these relationships are selected")
    useful_relationships: list[str] = Field(description="Selected relationship lines, each includes [id] prefix")


query_prompt_one_shot_input = """I will provide you with a list of relationship descriptions. Your task is to select 3 relationships that may be useful to answer the given question. Please return a JSON object containing your thought process and a list of the selected relationships in order of their relevance.

Question:
When was the mother of the leader of the Third Crusade born?

Relationship descriptions:
[1] Eleanor was born in 1122.
[2] Eleanor married King Louis VII of France.
[3] Eleanor was the Duchess of Aquitaine.
[4] Eleanor participated in the Second Crusade.
[5] Eleanor had eight children.
[6] Eleanor was married to Henry II of England.
[7] Eleanor was the mother of Richard the Lionheart.
[8] Richard the Lionheart was the King of England.
[9] Henry II was the father of Richard the Lionheart.
[10] Henry II was the King of England.
[11] Richard the Lionheart led the Third Crusade.
"""

query_prompt_one_shot_output = """{"thought_process": "To answer the question about the birth of the mother of the leader of the Third Crusade, I first need to identify who led the Third Crusade and then determine who his mother was. After identifying his mother, I can look for the relationship that mentions her birth.", "useful_relationships": ["[11] Richard the Lionheart led the Third Crusade", "[7] Eleanor was the mother of Richard the Lionheart", "[1] Eleanor was born in 1122"]}"""

query_prompt_template = """Question:
{question}

Relationship descriptions:
{relation_des_str}
"""


def rerank_relations(query: str, relation_candidate_texts: list[str], relation_candidate_ids: list[int]) -> list[int]:
    relation_des_str = "\n".join(
        f"[{rid}] {txt}" for rid, txt in zip(relation_candidate_ids, relation_candidate_texts)
    ).strip()

    prompt = ChatPromptTemplate.from_messages(
        [
            HumanMessage(query_prompt_one_shot_input),
            AIMessage(query_prompt_one_shot_output),
            ("human", query_prompt_template),
        ]
    )

    rerank_llm = llm.with_structured_output(RerankResult, method="json_schema", strict=True)
    res = (prompt | rerank_llm).invoke({"question": query, "relation_des_str": relation_des_str})

    picked_ids: list[int] = []
    for line in res.useful_relationships:
        left = line.find("[")
        right = line.find("]")
        picked_ids.append(int(line[left + 1 : right]))
    return picked_ids

In [ ]:
rerank_relation_ids = rerank_relations(
    query,
    relation_candidate_texts=relation_candidate_texts,
    relation_candidate_ids=relation_candidate_ids,
)

rerank_relation_ids

[12, 0, 1]

## 8) 用 rerank 后的关系拿回最终 passages，并对照 naive RAG


In [ ]:
final_passage_ids: set[int] = set()
for rid in rerank_relation_ids:
    final_passage_ids.update(relationid_2_passageids[rid])

final_passages = [passages[i] for i in sorted(final_passage_ids)]
final_passages

['Jakob Bernoulli (1654–1705): Jakob was one of the earliest members of the Bernoulli family to gain prominence in mathematics. He made significant contributions to calculus, particularly in the development of the theory of probability. He is known for the Bernoulli numbers and the Bernoulli theorem, a precursor to the law of large numbers. He was the older brother of Johann Bernoulli, another influential mathematician, and the two had a complex relationship that involved both collaboration and rivalry.',
 'Daniel Bernoulli (1700–1782): The son of Johann Bernoulli, Daniel made major contributions to fluid dynamics, probability, and statistics. He is most famous for Bernoulli’s principle, which describes the behavior of fluid flow and is fundamental to the understanding of aerodynamics.']

In [ ]:
naive_res = milvus_client.search(
    collection_name=passage_col_name,
    data=[query_embedding],
    limit=3,
    output_fields=["id", "text"],
)[0]

[hit["entity"]["text"] for hit in naive_res]

['Leonhard Euler (1707–1783) was one of the greatest mathematicians of all time, and his relationship with the Bernoulli family was significant. Euler was born in Basel and was a student of Johann Bernoulli, who recognized his exceptional talent and mentored him in mathematics. Johann Bernoulli’s influence on Euler was profound, and Euler later expanded upon many of the ideas and methods he learned from the Bernoullis.',
 'Johann Bernoulli (1667–1748): Johann, Jakob’s younger brother, was also a major figure in the development of calculus. He worked on infinitesimal calculus and was instrumental in spreading the ideas of Leibniz across Europe. Johann also contributed to the calculus of variations and was known for his work on the brachistochrone problem, which is the curve of fastest descent between two points.',
 'Jakob Bernoulli (1654–1705): Jakob was one of the earliest members of the Bernoulli family to gain prominence in mathematics. He made significant contributions to calculus, 

## 9) 用 GraphRAG passages 生成最终答案


In [ ]:
context = "\n\n".join(f"[P{i+1}] {p}" for i, p in enumerate(final_passages))

answer_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a careful assistant. Answer strictly using the provided passages."),
        ("human", "Question:\n{question}\n\nPassages:\n{context}\n\nAnswer:"),
    ]
)

answer = (answer_prompt | llm).invoke({"question": query, "context": context})
answer.content

"The son of Euler's teacher, Johann Bernoulli, is Daniel Bernoulli. He made major contributions to fluid dynamics, probability, and statistics, and is most famous for Bernoulli’s principle, which describes the behavior of fluid flow and is fundamental to the understanding of aerodynamics."

I0621 17:59:25.578892 2207637 chttp2_transport.cc:1376] ipv4:127.0.0.1:41981: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {http2_error:11}
E0621 17:59:25.579031 2207637 chttp2_transport.cc:1408] ipv4:127.0.0.1:41981: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 10000ms
